## Załadowanie bibliotek

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import re
import string
import random

# Dla BoW
from sklearn.feature_extraction.text import CountVectorizer

# Dla Word2Vec
from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download
import nltk
from nltk.tokenize import word_tokenize

# Dla BERT
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

nltk.download("punkt")

# === KONFIGURACJA ===
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jakub\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Dane - recenzje IMDB

In [3]:
def clean_text(text):
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.lower()

def load_and_prepare_data():
    raw_dataset = load_dataset("imdb")

    df_train_full = pd.DataFrame(raw_dataset["train"])
    df_test = pd.DataFrame(raw_dataset["test"])

    # Czyszczenie
    df_train_full["clean_review"] = df_train_full["text"].apply(clean_text)
    df_test["clean_review"] = df_test["text"].apply(clean_text)

    # Podział train/val
    train_df, val_df = train_test_split(df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=SEED)

    # 🔻 OGRANICZENIE liczby próbek (dla BERT-a)
    train_df = train_df.sample(2000, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(500, random_state=SEED).reset_index(drop=True)
    test_df = df_test.sample(1000, random_state=SEED).reset_index(drop=True)

    return train_df, val_df, test_df



# Datasets
### BoW + Feedforward NN

In [4]:
class BoWDataset(Dataset):
    def __init__(self, texts, labels, vectorizer=None, fit=False):
        self.texts = texts
        self.labels = labels

        if fit:
            self.vectorizer = CountVectorizer(max_features=10000)
            self.features = self.vectorizer.fit_transform(texts).toarray()
        else:
            self.vectorizer = vectorizer
            self.features = self.vectorizer.transform(texts).toarray()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

    def get_vectorizer(self):
        return self.vectorizer

### Word2Vec + LSTM

In [5]:
from tqdm.auto import tqdm # Add this import for the progress bar

class Word2VecDataset(Dataset):
    def __init__(self, texts, labels, w2v_model, max_len=300):
        if isinstance(labels, pd.Series):
            self.labels = torch.tensor(labels.tolist(), dtype=torch.float32)
        else:
            self.labels = torch.tensor(labels, dtype=torch.float32)
        
        self.w2v = w2v_model
        self.dim = w2v_model.vector_size
        self.max_len = max_len

        print(f"Preprocessing {len(texts)} samples for Word2VecDataset...")
        self.features = []
        # This loop with tqdm will run once and show a progress bar
        for text in tqdm(texts):
            tokens = word_tokenize(text)
            vectors = []

            for token in tokens:
                if token in self.w2v:
                    vectors.append(self.w2v[token])
                if len(vectors) == self.max_len:
                    break
            
            if len(vectors) < self.max_len:
                pad_len = self.max_len - len(vectors)
                vectors.extend([np.zeros(self.dim)] * pad_len)
            
            self.features.append(torch.tensor(np.array(vectors), dtype=torch.float32))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Now we just return the pre-processed tensor
        return self.features[idx], self.labels[idx]

### BERT

In [6]:
class BERTDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


# DataLoaders

In [7]:
def prepare_dataloaders(model_type, train_df, val_df, batch_size=32):
    if model_type == "bow":
        # === BoW ===
        train_ds = BoWDataset(train_df["clean_review"], train_df["label"], fit=True)
        vectorizer = train_ds.get_vectorizer()
        # The line below is now fixed
        val_ds = BoWDataset(val_df["clean_review"], val_df["label"], vectorizer)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size)
        return train_loader, val_loader, vectorizer

    elif model_type == "word2vec":
        # === Word2Vec ===
        print("[prepare_dataloaders] Ładowanie pretrenowanego Word2Vec...")
        model_path = hf_hub_download(repo_id="NathaNn1111/word2vec-google-news-negative-300-bin",
                                     filename="GoogleNews-vectors-negative300.bin")
        w2v = KeyedVectors.load_word2vec_format(model_path, binary=True)
        print("[prepare_dataloaders] Załadowano model Word2Vec.")

        train_ds = Word2VecDataset(train_df["clean_review"], train_df["label"], w2v)
        val_ds = Word2VecDataset(val_df["clean_review"], val_df["label"], w2v)

        # Set num_workers=0 to avoid issues, especially on Windows
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=0)
        print("[prepare_dataloaders] Utworzono DataLoader Word2Vec.")
        return train_loader, val_loader, w2v

    elif model_type == "bert":
        # === BERT ===
        tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

        train_ds = BERTDataset(train_df["clean_review"].tolist(), train_df["label"].tolist(), tokenizer)
        val_ds = BERTDataset(val_df["clean_review"].tolist(), val_df["label"].tolist(), tokenizer)

        return train_ds, val_ds, tokenizer

    else:
        raise ValueError(f"Unsupported model_type: {model_type}")


# Models
### BoW + Feedforward NN

In [8]:
class BoWModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=100):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x).squeeze()


### Word2Vec + LSTM

In [9]:
class LSTMModel(nn.Module):
    def __init__(self, embedding_dim=300, hidden_dim=128, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=embedding_dim,
                            hidden_size=hidden_dim,
                            num_layers=num_layers,
                            batch_first=True)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)  # hidden: [num_layers, batch, hidden_dim]
        return self.classifier(hidden[-1]).squeeze()


### BERT 

In [10]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Train and Evaluate

### Train

In [11]:
def train_model(model, train_loader, val_loader, num_epochs=5, lr=1e-3):
    model = model.to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        # --- trening ---
        model.train()
        train_loss, train_correct = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            preds = model(x)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_correct += ((preds >= 0.5).float() == y).sum().item()

        train_acc = train_correct / len(train_loader.dataset)

        # --- walidacja ---
        model.eval()
        val_loss, val_correct = 0.0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds = model(x)
                loss = criterion(preds, y)

                val_loss += loss.item()
                val_correct += ((preds >= 0.5).float() == y).sum().item()

        val_acc = val_correct / len(val_loader.dataset)

        print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pth")
            print("Model saved.")

    return model


### Evaluate

In [12]:
def evaluate_model(model, test_loader):
    model = model.to(DEVICE)
    model.eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            preds = model(x)
            preds = (preds >= 0.5).int().cpu().numpy()
            y_true.extend(y.numpy())
            y_pred.extend(preds)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


### Train and evaluate transformer (BERT)

In [13]:
from transformers import TrainingArguments, Trainer

def train_and_evaluate_bert(train_dataset, val_dataset):
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

    training_args = TrainingArguments(
        output_dir="./bert_output",
        num_train_epochs=1,  # 🔻 tylko 1 epoka
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",  # zamiast deprecated evaluation_strategy
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
        logging_dir="./logs",
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = logits.argmax(axis=1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "precision": precision_score(labels, preds),
            "recall": recall_score(labels, preds),
            "f1": f1_score(labels, preds),
        }

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    metrics = trainer.evaluate()
    return trainer.model, metrics


# Main function to train and evaluate models

In [14]:
def run_pipeline(model_type):
    print(f"\n Running pipeline for: {model_type.upper()}")

    train_df, val_df, test_df = load_and_prepare_data()

    if model_type == "bert":
        train_ds, val_ds, tokenizer = prepare_dataloaders("bert", train_df, val_df)
        model, metrics = train_and_evaluate_bert(train_ds, val_ds)

    else:
        train_loader, val_loader, vectorizer_or_w2v = prepare_dataloaders(model_type, train_df, val_df)

        if model_type == "bow":
            test_ds = BoWDataset(test_df["clean_review"], test_df["label"], vectorizer_or_w2v)
            test_loader = DataLoader(test_ds, batch_size=32)
            model = BoWModel(input_dim=len(vectorizer_or_w2v.get_feature_names_out()))
        elif model_type == "word2vec":
            print("Preparing test_ds")
            test_ds = Word2VecDataset(test_df["clean_review"], test_df["label"], vectorizer_or_w2v)
            print("Preparing test_loader")
            test_loader = DataLoader(test_ds, batch_size=32)
            print("Preparing model")
            model = LSTMModel()

        model = train_model(model, train_loader, val_loader, num_epochs=5)
        metrics = evaluate_model(model, test_loader)

    print(f"\n {model_type.upper()} DONE! Metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    return metrics


In [33]:
metrics_bow = run_pipeline("bow")


 Running pipeline for: BOW
Epoch 1/5 | Train Acc: 0.8527 | Val Acc: 0.8862
Model saved.
Epoch 2/5 | Train Acc: 0.9235 | Val Acc: 0.8914
Model saved.
Epoch 3/5 | Train Acc: 0.9506 | Val Acc: 0.8884
Epoch 4/5 | Train Acc: 0.9718 | Val Acc: 0.8836
Epoch 5/5 | Train Acc: 0.9850 | Val Acc: 0.8850
Accuracy:  0.8628
Precision: 0.8469
Recall:    0.8858
F1-score:  0.8659

 BOW DONE! Metrics:
  accuracy: 0.8628
  precision: 0.8469
  recall: 0.8858
  f1: 0.8659


In [18]:
metrics_w2v = run_pipeline("word2vec")


 Running pipeline for: WORD2VEC
[prepare_dataloaders] Ładowanie pretrenowanego Word2Vec...
[prepare_dataloaders] Załadowano model Word2Vec.
Preprocessing 20000 samples for Word2VecDataset...


100%|██████████| 20000/20000 [00:28<00:00, 714.17it/s]


Preprocessing 5000 samples for Word2VecDataset...


100%|██████████| 5000/5000 [00:08<00:00, 604.80it/s]


[prepare_dataloaders] Utworzono DataLoader Word2Vec.
Preparing test_ds
Preprocessing 25000 samples for Word2VecDataset...


100%|██████████| 25000/25000 [02:01<00:00, 205.82it/s]


Preparing test_loader
Preparing model
Epoch 1/5 | Train Acc: 0.5076 | Val Acc: 0.5200
Model saved.
Epoch 2/5 | Train Acc: 0.5684 | Val Acc: 0.5468
Model saved.
Epoch 3/5 | Train Acc: 0.5968 | Val Acc: 0.5000
Epoch 4/5 | Train Acc: 0.5111 | Val Acc: 0.5000
Epoch 5/5 | Train Acc: 0.5731 | Val Acc: 0.6790
Model saved.
Accuracy:  0.6858
Precision: 0.7260
Recall:    0.5967
F1-score:  0.6550

 WORD2VEC DONE! Metrics:
  accuracy: 0.6858
  precision: 0.7260
  recall: 0.5967
  f1: 0.6550


In [15]:
metrics_bert = run_pipeline("bert")


 Running pipeline for: BERT


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 0/63 [00:00<?, ?it/s]c:\Projects\Studia\venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
 16%|█▌        | 10/63 [05:55<32:03, 36.29s/it]

{'loss': 0.711, 'grad_norm': 0.9997637867927551, 'learning_rate': 4.2063492063492065e-05, 'epoch': 0.16}


 32%|███▏      | 20/63 [11:19<23:05, 32.23s/it]

{'loss': 0.652, 'grad_norm': 3.2250723838806152, 'learning_rate': 3.412698412698413e-05, 'epoch': 0.32}


...

{'eval_loss': 0.34755048155784607, 'eval_accuracy': 0.856, 'eval_precision': 0.9036697247706422, 'eval_recall': 0.7943548387096774, 'eval_f1': 0.8454935622317596, 'eval_runtime': 128.0852, 'eval_samples_per_second': 3.904, 'eval_steps_per_second': 0.125, 'epoch': 1.0}


100%|██████████| 63/63 [36:53<00:00, 35.13s/it]
c:\Projects\Studia\venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'train_runtime': 2213.1817, 'train_samples_per_second': 0.904, 'train_steps_per_second': 0.028, 'train_loss': 0.4838095165434338, 'epoch': 1.0}


100%|██████████| 16/16 [01:50<00:00,  6.89s/it]


 BERT DONE! Metrics:
  eval_loss: 0.3476
  eval_accuracy: 0.8560
  eval_precision: 0.9037
  eval_recall: 0.7944
  eval_f1: 0.8455
  eval_runtime: 118.2404
  eval_samples_per_second: 4.2290
  eval_steps_per_second: 0.1350
  epoch: 1.0000
